In [0]:
silver_df = spark.table("silver_customer_products")

display(silver_df.limit(5))

department_id,aisle_id,product_id,order_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
19,107,16589,2235,1,1,177129,prior,22,6,14,4.0,Plantain Chips,chips pretzels,snacks
4,16,28842,2235,2,1,177129,prior,22,6,14,4.0,Bunched Cilantro,fresh herbs,produce
4,24,13176,3923,1,1,148615,prior,32,1,17,22.0,Bag of Organic Bananas,fresh fruits,produce
4,123,21903,3923,2,1,148615,prior,32,1,17,22.0,Organic Baby Spinach,packaged vegetables fruits,produce
4,24,42633,3923,3,0,148615,prior,32,1,17,22.0,Red Delicious Apple,fresh fruits,produce


In [0]:
print("Total rows")

print(silver_df.count())


Total rows
32434489


In [0]:
print("Distinct orders")

print(
    silver_df
    .select("order_id")
    .distinct()
    .count()
)

Distinct orders
3214874


In [0]:
print("Min Order ID")

display(
    silver_df.selectExpr(
        "min(order_id)"
    )
)


print("Max Order ID")

display(
    silver_df.selectExpr(
        "max(order_id)"
    )
)

Min Order ID


min(order_id)
2


Max Order ID


max(order_id)
3421083


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("order_id")

orders_batch = (
    silver_df
    .select("order_id")
    .distinct()
    .orderBy("order_id")
    .withColumn(
        "row_num",
        row_number().over(window)
    )
    .withColumn(
        "batch_no",
        ((row_number().over(window)-1)/1000).cast("int")
    )
)

display(orders_batch.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,row_num,batch_no
2,1,0
3,2,0
4,3,0
5,4,0
6,5,0
7,6,0
8,7,0
9,8,0
10,9,0
11,10,0


In [0]:
orders_batch.selectExpr(
    "max(batch_no)"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+
|max(batch_no)|
+-------------+
|         3214|
+-------------+



In [0]:
stream_path = "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders"

In [0]:
batch0_ids = (
    orders_batch
    .filter("batch_no = 0")
    .select("order_id")
)

batch0 = (
    silver_df
    .join(
        batch0_ids,
        on="order_id"
    )
)

display(batch0.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,department_id,aisle_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
129,4,24,13176,1,0,169467,prior,7,4,18,3.0,Bag of Organic Bananas,fresh fruits,produce
129,16,120,35628,2,1,169467,prior,7,4,18,3.0,Organic Strawberry Smoothie,yogurt,dairy eggs
129,7,64,33454,3,1,169467,prior,7,4,18,3.0,Power-C Dragonfruit Vitamin Water Drink,energy sports drinks,beverages
129,4,83,15649,4,1,169467,prior,7,4,18,3.0,Baby Seedless Cucumbers,fresh vegetables,produce
129,19,61,26634,5,0,169467,prior,7,4,18,3.0,"Cookies, Chocolate Chip Walnut",cookies cakes,snacks
129,12,122,13198,6,1,169467,prior,7,4,18,3.0,85% Lean Ground Beef,meat counter,meat seafood
129,4,83,41950,7,0,169467,prior,7,4,18,3.0,Organic Tomato Cluster,fresh vegetables,produce
129,7,98,24850,8,1,169467,prior,7,4,18,3.0,Organic Super Fruit Punch Juice Drink,juice nectars,beverages
129,8,40,17471,9,1,169467,prior,7,4,18,3.0,Small Dog Biscuits,dog food care,pets
129,19,50,5194,10,1,169467,prior,7,4,18,3.0,Organic Bunny Fruit Snacks Berry Patch,fruit vegetable snacks,snacks


In [0]:
batch0.write.mode("overwrite").csv(

    f"{stream_path}/batch_0"

)

print("Batch 0 sent")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Batch 0 sent


In [0]:
dbutils.fs.rm(stream_path,True)
dbutils.fs.mkdirs(stream_path)

True

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv/_SUCCESS,_SUCCESS,0,1782537454000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv/_committed_243814046065352060,_committed_243814046065352060,112,1782537454000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv/_started_243814046065352060,_started_243814046065352060,0,1782537451000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_000.csv/part-00000-tid-243814046065352060-5f4dd721-bfce-4f2f-8f40-461dc48ca2ef-444-1-c000.csv,part-00000-tid-243814046065352060-5f4dd721-bfce-4f2f-8f40-461dc48ca2ef-444-1-c000.csv,966088,1782537453000


In [0]:
import time

for i in range(15,20):

    ids = (
        orders_batch
        .filter(f"batch_no={i}")
        .select("order_id")
    )

    batch = silver_df.join(ids,"order_id")


    temp_path = f"/Volumes/workspace/default/e-commerce_recommendation/temp"


    dbutils.fs.rm(temp_path,True)


    (
        batch
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header",True)
        .csv(temp_path)
    )


    files = dbutils.fs.ls(temp_path)

    csv_file = [f.path for f in files if f.name.endswith(".csv")][0]


    target = f"{stream_path}/orders_{i:03d}.csv"


    dbutils.fs.mv(
        csv_file,
        target
    )


    print(target)

    time.sleep(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_015.csv


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_016.csv


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_017.csv
/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_018.csv
/Volumes/workspace/default/e-commerce_recommendation/incoming_orders/orders_019.csv
